In [1]:
import warnings
import joblib
import os, time
import numpy as np
import pandas as pd
import geohash2
# ML libs
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import MinMaxScaler
import lightgbm as lgb
from xgboost import XGBRegressor

# TS libs
from statsmodels.tsa.statespace.sarimax import SARIMAX
try:
    from prophet import Prophet
    HAS_PROPHET = True
except Exception:
    HAS_PROPHET = False

# DL libs
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# Geospatial
from haversine import haversine

In [2]:
veri = pd.read_csv("C:/Users/kerem/Desktop/Akıllı Trafik/Modeller/proje.csv")
veri = veri.copy()
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.float_format', lambda  x: '%.3f' % x)
pd.set_option('display.width', 1000)
warnings.filterwarnings('ignore')

In [3]:
# ---------------- USER HYPERPARAMS ----------------
HIST     = 35          # input window (time steps)
HORIZON  = 12          # output horizon (we will meta on first horizon step)
BATCH    = 32
EPOCHS   = 12          # keep small for OOF pilot; increase for final training
LR       = 1e-3
N_FOLDS  = 3           # expanding-window folds for OOF (keep small for faster)
DEVICE   = torch.device("cuda" if torch.cuda.is_available() else "cpu")
OUT_DIR  = "hybrid_oof_outputs"
os.makedirs(OUT_DIR, exist_ok=True)
RND = 42
np.random.seed(RND)
torch.manual_seed(RND)

print("Device:", DEVICE)

Device: cuda


In [4]:
# ------------------ 1) Basic cleaning & pivot -> build (T,N,C) tensor ------------------
# Required columns: DATE, GEOHASH_SHORT, DENSITY
veri = veri.copy()
veri['DATE'] = pd.to_datetime(veri['DATE'], errors='coerce')
veri = veri.dropna(subset=['DATE','GEOHASH_SHORT','DENSITY']).reset_index(drop=True)

# ensure TIME_PERIOD -> period_code (0..4) if present as TIME_PERIOD or period_code column
if 'period_code' not in veri.columns:
    if 'TIME_PERIOD' in veri.columns:
        veri['period_code'] = veri['TIME_PERIOD'].astype('category').cat.codes
    else:
        # fallback default 0
        veri['period_code'] = 0

# ensure rain_encoded exists if possible
if 'rain_encoded' not in veri.columns and 'rainfall_condition' in veri.columns:
    veri['rain_encoded'] = veri['rainfall_condition'].astype('category').cat.codes
elif 'rain_encoded' not in veri.columns:
    veri['rain_encoded'] = 0

# Optional features list (first channel MUST be DENSITY)
base_features = ['DENSITY']
optional_features = ['AVERAGE_SPEED','NUMBER_OF_VEHICLES','rain_encoded']
features_present = [f for f in base_features + optional_features if f in veri.columns]

# Node list and index
node_list = list(veri['GEOHASH_SHORT'].drop_duplicates())
node_list.sort()
N = len(node_list)
print(f"N nodes: {N}")

# Pivot helper: pivot(index=(DATE,period_code), columns=GEOHASH_SHORT, values=feature)
def pivot_feature_matrix(veri, node_list, feature):
    piv = veri.pivot_table(index=['DATE','period_code'], columns='GEOHASH_SHORT', values=feature)
    piv = piv.reindex(columns=node_list)
    piv = piv.sort_index().fillna(method='ffill').fillna(method='bfill').fillna(0.0)
    return piv

# Build matrices
mats = []
p_density = pivot_feature_matrix(veri, node_list, 'DENSITY')
index_ref = p_density.index  # MultiIndex (DATE,period_code)
mats.append(p_density.values.astype(np.float32))   # channel 0

for feat in optional_features:
    if feat in veri.columns:
        piv = pivot_feature_matrix(veri, node_list, feat)
        # align to density index
        piv = piv.reindex(index_ref).fillna(method='ffill').fillna(method='bfill').fillna(0.0)
        mats.append(piv.values.astype(np.float32))

# period one-hot channels
period_idx = np.array([int(x[1]) for x in index_ref])
n_periods = max(period_idx.max()+1, 5)  # assume up to 5
for p in range(n_periods):
    vec = (period_idx == p).astype(np.float32)
    mat = np.tile(vec.reshape(-1,1), (1,N))
    mats.append(mat.astype(np.float32))

# stack -> data_orig (T,N,C)
data_orig = np.stack(mats, axis=-1)   # (T, N, C)
T, N_check, C = data_orig.shape
assert N_check == N
print("Tensor shape (T,N,C):", data_orig.shape)

# density scaler (for DL inverse)
density_arr = data_orig[:,:,0].reshape(-1,1)
density_scaler = MinMaxScaler(feature_range=(0,1))
density_scaler.fit(density_arr)

# per-channel scalers for DL input
scalers = []
data_scaled = np.zeros_like(data_orig, dtype=np.float32)
for c in range(C):
    arr = data_orig[:,:,c].reshape(-1,1)
    sc = MinMaxScaler(feature_range=(0,1))
    sc.fit(arr)
    data_scaled[:,:,c] = sc.transform(arr).reshape(T,N)
    scalers.append(sc)

# Create sequences for DL
def create_sequences(data_arr, hist, horizon):
    T = data_arr.shape[0]
    Xs, Ys = [], []
    for s in range(0, T - hist - horizon + 1):
        x = data_arr[s:s+hist]            # (hist, N, C)
        y = data_arr[s+hist:s+hist+horizon, :, 0]  # density channel target (horizon,N)
        Xs.append(x)
        Ys.append(y)
    Xs = np.stack(Xs).astype(np.float32) if len(Xs)>0 else np.zeros((0,hist,N,C), dtype=np.float32)
    Ys = np.stack(Ys).astype(np.float32) if len(Ys)>0 else np.zeros((0,horizon,N), dtype=np.float32)
    return Xs, Ys

X_dl, Y_dl = create_sequences(data_scaled, HIST, HORIZON)   # scaled for DL
X_orig, Y_orig = create_sequences(data_orig, HIST, HORIZON)  # original units for tabular/SARIMAX/Prophet
S = len(X_dl)
print("Samples S:", S, "X shape:", X_dl.shape, "Y shape:", Y_dl.shape)

if S == 0:
    raise ValueError("No sequences created. Reduce HIST/HORIZON or check T.")

# Sample -> time mapping: sample s target time index = s + HIST (index into index_ref)
sample_target_timeidx = np.arange(S) + HIST  # values in [HIST, T-horizon]

N nodes: 199
Tensor shape (T,N,C): (1368, 199, 9)
Samples S: 1322 X shape: (1322, 35, 199, 9) Y shape: (1322, 12, 199)


In [5]:
# ------------------ 2) Fold generator (expanding-window style on time axis) ------------------
def generate_time_folds(T, n_folds=3, min_train_ratio=0.2):
    folds = []
    for k in range(1, n_folds+1):
        train_end = int(T * (min_train_ratio + (k-1) * (1-min_train_ratio)/n_folds))
        val_end   = int(T * (min_train_ratio + k * (1-min_train_ratio)/n_folds))
        # train samples: s where s+hist <= train_end
        # val samples: s where s+hist > train_end and s+hist <= val_end
        folds.append((train_end, val_end))
    return folds

fold_bounds = generate_time_folds(T, n_folds=N_FOLDS, min_train_ratio=0.2)
fold_bounds

# For later mapping sample indices per fold:
fold_sample_indices = []
for (train_end, val_end) in fold_bounds:
    train_idx = [s for s in range(S) if (s + HIST) <= train_end]
    val_idx   = [s for s in range(S) if (s + HIST) > train_end and (s + HIST) <= val_end]
    fold_sample_indices.append((np.array(train_idx, dtype=int), np.array(val_idx, dtype=int)))
    print(f"Fold train_end={train_end} val_end={val_end} -> train samples {len(train_idx)} val samples {len(val_idx)}")

Fold train_end=273 val_end=638 -> train samples 239 val samples 365
Fold train_end=638 val_end=1003 -> train samples 604 val samples 365
Fold train_end=1003 val_end=1368 -> train samples 969 val samples 353


In [6]:
# ------------------ 3) Model definitions (light versions) ------------------

# --- Simple LSTM (flatten nodes*channels per time-step) ---
class LSTM_Flat(nn.Module):
    def __init__(self, hist, N, C, hidden=128, layers=2, horizon=HORIZON):
        super().__init__()
        self.input_dim = N*C
        self.lstm = nn.LSTM(input_size=self.input_dim, hidden_size=hidden, num_layers=layers, batch_first=True, dropout=0.2)
        self.fc = nn.Linear(hidden, horizon * N)
        self.horizon = horizon
        self.N = N
    def forward(self, x):
        # x: (B, hist, N, C)
        B = x.size(0)
        x = x.view(B, x.size(1), -1)  # (B, hist, N*C)
        out, _ = self.lstm(x)
        last = out[:, -1, :]
        y = self.fc(last).view(B, self.horizon, self.N)
        return y

# --- Simple STGCN_Multi (as earlier but lightweight) ---
class TemporalConvLayer(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3):
        super().__init__()
        pad = (kernel_size - 1)//2
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size=(1,kernel_size), padding=(0,pad))
        self.bn = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU()
    def forward(self, x):
        x = self.conv(x)
        x = self.bn(x)
        x = self.relu(x)
        return x

class GraphConvLayer(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.fc = nn.Linear(in_ch, out_ch)
        self.relu = nn.ReLU()
        self.bn = nn.BatchNorm2d(out_ch)
    def forward(self, x, A):
        # x: (B, C, N, T)
        B, C, N, T = x.shape
        xt = x.permute(0,3,2,1).contiguous()  # (B, T, N, C)
        out_list = []
        A = A.to(x.device)
        for b in range(B):
            xbt = xt[b]  # (T, N, C)
            out_bt = []
            for t in range(T):
                xtt = xbt[t]  # (N, C)
                axt = A @ xtt
                out_bt.append(axt)
            out_bt = torch.stack(out_bt, dim=0)
            out_list.append(out_bt)
        out = torch.stack(out_list, dim=0)
        out = self.fc(out)
        out = out.permute(0,3,2,1).contiguous()
        out = self.relu(out)
        out = self.bn(out)
        return out

class STGCNBlock(nn.Module):
    def __init__(self, in_ch, mid_ch, out_ch):
        super().__init__()
        self.t1 = TemporalConvLayer(in_ch, mid_ch)
        self.g = GraphConvLayer(mid_ch, mid_ch)
        self.t2 = TemporalConvLayer(mid_ch, out_ch)
        if in_ch == out_ch:
            self.res = lambda x: x
        else:
            self.res = nn.Conv2d(in_ch, out_ch, kernel_size=(1,1))
        self.relu = nn.ReLU()
    def forward(self, x, A):
        r = self.res(x)
        x = self.t1(x)
        x = self.g(x, A)
        x = self.t2(x)
        x = x + r
        x = self.relu(x)
        return x

class STGCN_Simple(nn.Module):
    def __init__(self, num_nodes, in_ch, A, blocks=[(16,32),(32,64)], pred_horizon=HORIZON):
        super().__init__()
        self.A = A   # A burada kaydediliyor

        layers = []
        cin = in_ch
        for (mid, out) in blocks:
            layers.append(STGCNBlock(cin, mid, out))
            cin = out
        self.blocks = nn.ModuleList(layers)

        self.final_conv = nn.Conv2d(cin, pred_horizon, kernel_size=(1,1))
        self.pred_horizon = pred_horizon

    def forward(self, x):
        # x: (B, hist, N, C)
        A = self.A
        # (B,C,N,T) formatına geçir
        x = x.permute(0,3,2,1).contiguous()

        # STGCN blokları
        for blk in self.blocks:
            x = blk(x, A)

        # Final conv (B, pred_horizon, N, T)
        out = self.final_conv(x)

        # Sadece son zaman adımı
        out = out[:, :, :, -1]  # -> (B, pred_horizon, N)
        return out

In [7]:
# ------------------ 4) Training / Predict helper for DL models ------------------
class SeqDatasetDL(Dataset):
    def __init__(self, X, Y):
        # X: (S, hist, N, C) scaled; Y: (S, horizon, N) scaled/orig depending on input
        self.X = torch.tensor(X, dtype=torch.float32)
        self.Y = torch.tensor(Y, dtype=torch.float32)
    def __len__(self): return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.Y[idx]

def fit_dl_model(model, X_train, Y_train, X_val=None, epochs=EPOCHS, batch=BATCH, lr=LR, device=DEVICE):
    model = model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    crit = nn.MSELoss()
    train_dl = DataLoader(SeqDatasetDL(X_train, Y_train), batch_size=batch, shuffle=True)
    val_dl = DataLoader(SeqDatasetDL(X_val[0], X_val[1]), batch_size=batch, shuffle=False) if X_val is not None else None
    best_state = None
    best_val = np.inf
    patience = 0
    for ep in range(1, epochs+1):
        model.train()
        tr_losses = []
        for xb, yb in train_dl:
            xb = xb.to(device); yb = yb.to(device)
            opt.zero_grad()
            preds = model(xb)
            loss = crit(preds[:,:, :], yb)   # compare multi-step but we use first step later
            loss.backward(); opt.step()
            tr_losses.append(loss.item())
        if val_dl is not None:
            model.eval(); val_losses=[]
            with torch.no_grad():
                for xb, yb in val_dl:
                    xb = xb.to(device); yb = yb.to(device)
                    preds = model(xb)
                    val_losses.append(crit(preds, yb).item())
            avg_val = np.mean(val_losses)
            if avg_val < best_val:
                best_val = avg_val; best_state = model.state_dict(); patience=0
                torch.save(best_state, os.path.join(OUT_DIR, f"best_{int(time.time())}.pth"))
            else:
                patience += 1
            if patience > 6:
                break
    if best_state is not None:
        model.load_state_dict(best_state)
    return model

def dl_predict(model, X):
    model = model.to(DEVICE)
    model.eval()
    ds = DataLoader(SeqDatasetDL(X, np.zeros((len(X), HORIZON, N), dtype=np.float32)), batch_size=BATCH, shuffle=False)
    preds = []
    with torch.no_grad():
        for xb, _ in ds:
            xb = xb.to(DEVICE)
            out = model(xb).cpu().numpy()
            preds.append(out)
    return np.concatenate(preds, axis=0)  # (S, horizon, N)

In [8]:
def build_tabular_features_from_X(data_orig, sample_indices, node_list, num_lags=[1,2,3]):
    rows = []
    T, N, C = data_orig.shape

    for s in sample_indices:
        # history window (hist, N, C)
        hist_window = data_orig[s : s + HIST]

        for node_i in range(N):

            d_series = hist_window[:, node_i, 0]   # (hist,)

            feats = {
                "density_last": float(d_series[-1]),
                "density_mean_hist": float(d_series.mean()),
                "density_std_hist": float(d_series.std()),
            }

            # lags
            for lag in num_lags:
                if lag <= len(d_series):
                    feats[f"d_lag_{lag}"] = float(d_series[-lag])
                else:
                    feats[f"d_lag_{lag}"] = float(d_series[0])

            feats["node_idx"] = node_i
            rows.append(feats)

    return pd.DataFrame(rows)

In [9]:
# ------------------ 6) SARIMAX / Prophet per-node fold predictor ------------------
def sarimax_forecast_for_fold(piv_density_orig, train_time_idx, val_time_idx):
    # piv_density_orig: (T,N) original density matrix (numpy)
    # train_time_idx, val_time_idx: ints on time axis where train covers [0:train_time_idx) and val covers [train_time_idx:val_time_idx)
    # We'll fit SARIMAX per node on y[:train_time_idx] and forecast steps = val_time_count
    val_count = val_time_idx - train_time_idx
    preds = np.full((val_count, N), np.nan, dtype=float)
    for i in range(N):
        series = piv_density_orig[:train_time_idx, i]
        try:
            m = SARIMAX(series, order=(1,1,1), seasonal_order=(1,0,0,5), enforce_stationarity=False, enforce_invertibility=False)
            r = m.fit(disp=False, maxiter=50)
            f = r.forecast(steps=val_count)
            preds[:, i] = np.asarray(f)
        except Exception as e:
            # fallback: mean
            preds[:, i] = np.nanmean(series)
    return preds  # shape (val_count, N)

def prophet_forecast_for_fold(piv_density_orig, period_vec, train_time_idx, val_time_idx):
    # returns (val_count, N) predictions if Prophet installed, else NaNs
    if not HAS_PROPHET:
        return np.full((val_time_idx-train_time_idx, N), np.nan)
    val_count = val_time_idx - train_time_idx
    preds = np.full((val_count, N), np.nan, dtype=float)
    dates = [ix[0] for ix in index_ref]
    for i in range(N):
        df_ts = pd.DataFrame({'ds': dates[:train_time_idx], 'y': piv_density_orig[:train_time_idx, i]})
        try:
            m = Prophet(daily_seasonality=False, weekly_seasonality=True, yearly_seasonality=False)
            # add period regressor
            df_reg = pd.DataFrame({'ds': dates[:train_time_idx], 'period_code': period_vec[:train_time_idx]})
            m.add_regressor('period_code')
            train_df = pd.concat([df_ts, df_reg['period_code']], axis=1)
            m.fit(train_df)
            future = pd.DataFrame({'ds': dates[train_time_idx:val_time_idx], 'period_code': period_vec[train_time_idx:val_time_idx]})
            fc = m.predict(future)
            preds[:, i] = fc['yhat'].values
        except Exception as e:
            preds[:, i] = np.nanmean(piv_density_orig[:train_time_idx, i])
    return preds

In [10]:
# ------------------ 7) OOF loops across folds ------------------
S_total = S
oof_preds = {
    'lstm': np.full((S_total, N), np.nan),
    'stgcn': np.full((S_total, N), np.nan),
    'lgbm': np.full((S_total, N), np.nan),
    'xgb': np.full((S_total, N), np.nan),
    'sarimax': np.full((S_total, N), np.nan),
    'prophet': np.full((S_total, N), np.nan)
}

# pivot density original for SARIMAX/Prophet
piv_density_orig = p_density.values.astype(float)  # (T,N)
period_vec = period_idx

for fold_i, (train_samples, val_samples) in enumerate(fold_sample_indices):
    print(f"\n=== FOLD {fold_i+1}/{len(fold_sample_indices)} | train samples {len(train_samples)} val samples {len(val_samples)} ===")
    if len(val_samples) == 0 or len(train_samples) == 0:
        print("Skipping empty fold")
        continue

    # time bounds
    train_time_end = (train_samples.max() + HIST) if len(train_samples)>0 else HIST
    val_time_end = (val_samples.max() + HIST) + 1  # exclusive
    val_time_start = (val_samples.min() + HIST)
    print("train_time_end idx:", train_time_end, "val_time range:", val_time_start, val_time_end)

    # ------- (A) Train LSTM (DL) -------
    Xtr = X_dl[train_samples]   # scaled
    Ytr = Y_dl[train_samples]   # first horizon (scaled density) shape (n_s, N)
    Xval = X_dl[val_samples]
    Yval = Y_dl[val_samples]
    # build model
    lstm_model = LSTM_Flat(HIST, N, C, hidden=128, layers=2, horizon=HORIZON)
    # train (we pass Y as full horizon to match interface but the loss computed on all horizons)
    lstm_model = fit_dl_model(lstm_model, Xtr, Ytr, X_val=(Xval, Yval), epochs=EPOCHS, batch=BATCH, lr=LR)
    preds_val = dl_predict(lstm_model, Xval)   # (n_val, horizon, N) scaled
    preds_first = preds_val[:,0,:]             # (n_val, N) scaled
    # inverse scale density channel back to original units
    preds_first_inv = density_scaler.inverse_transform(preds_first.reshape(-1,1)).reshape(preds_first.shape)
    oof_preds['lstm'][val_samples, :] = preds_first_inv
    print("LSTM oof filled")

    # ------- (B) Train STGCN -------
    # adjacency building from nodes coords (try from df lat/lon)
    try:
        nodes_df = veri[['GEOHASH_SHORT','lat','lon']].drop_duplicates().set_index('GEOHASH_SHORT').loc[node_list]
        coords = nodes_df[['lat','lon']].values
    except Exception:
        # decode from geohash
        lat_list=[]; lon_list=[]
        for gh in node_list:
            try:
                lat, lon = geohash2.decode(gh)
            except:
                lat, lon = (0.0, 0.0)
            lat_list.append(lat); lon_list.append(lon)
        coords = np.stack([lat_list, lon_list], axis=1)
    # build adjacency
    def build_adj(coords, k=8):
        Nn = coords.shape[0]
        dist = np.zeros((Nn,Nn))
        for i in range(Nn):
            for j in range(i+1, Nn):
                d = haversine((coords[i,0], coords[i,1]), (coords[j,0], coords[j,1]))
                dist[i,j]=d; dist[j,i]=d
        A = np.zeros((Nn,Nn))
        for i in range(Nn):
            idx = np.argsort(dist[i])[1:k+1]
            A[i, idx] = 1
        A = np.maximum(A, A.T)
        np.fill_diagonal(A, 0)
        # normalize
        A = A + np.eye(Nn)
        deg = np.sum(A, axis=1)
        d_inv = np.power(deg, -0.5)
        d_inv[np.isinf(d_inv)] = 0.
        D_inv = np.diag(d_inv)
        A_norm = D_inv @ A @ D_inv
        return torch.tensor(A_norm, dtype=torch.float32, device=DEVICE)
    A_norm_torch = build_adj(coords, k=min(8, N-1))

    stgcn_model = STGCN_Simple(num_nodes=N, in_ch=C, A=A_norm_torch, blocks=[(16,32),(32,64)], pred_horizon=HORIZON)
    stgcn_model = fit_dl_model(stgcn_model, Xtr, Ytr, X_val=(Xval, Yval), epochs=EPOCHS, batch=BATCH, lr=LR)
    preds_val = dl_predict(stgcn_model, Xval)  # (n_val, horizon, N) scaled
    preds_first = preds_val[:,0,:]
    preds_first_inv = density_scaler.inverse_transform(preds_first.reshape(-1,1)).reshape(preds_first.shape)
    oof_preds['stgcn'][val_samples,:] = preds_first_inv
    print("STGCN oof filled")

    # ------- (C) Tabular models (LGBM, XGB) -------
    # Build tabular features per sample-node (use original units)
    feat_df_train = build_tabular_features_from_X(data_orig, train_samples, node_list)
    y_tab_train = []
    for s in train_samples:
        y_tab_train.append(Y_orig[s][0])  # first horizon (N,)
    y_tab_train = np.vstack(y_tab_train)  # (n_train, N)
    y_tab_train = y_tab_train.reshape(-1) # flatten
    X_tab_train = feat_df_train.values

    feat_df_val = build_tabular_features_from_X(data_orig, val_samples, node_list)
    X_tab_val = feat_df_val.values

    # LightGBM
    lgbm = lgb.LGBMRegressor(n_estimators=200, learning_rate=0.05, random_state=RND)
    lgbm.fit(X_tab_train, y_tab_train)
    preds_tab_val = lgbm.predict(X_tab_val).reshape(len(val_samples), N)
    oof_preds['lgbm'][val_samples,:] = preds_tab_val
    print("LGBM oof filled")

    # XGBoost
    xgbm = XGBRegressor(n_estimators=200, learning_rate=0.05, random_state=RND, verbosity=0)
    xgbm.fit(X_tab_train, y_tab_train)
    preds_tab_val = xgbm.predict(X_tab_val).reshape(len(val_samples), N)
    oof_preds['xgb'][val_samples,:] = preds_tab_val
    print("XGB oof filled")

    # ------- (D) SARIMAX per-node forecast -------
    sar_preds = sarimax_forecast_for_fold(piv_density_orig, train_time_end, val_time_end)
    # sar_preds shape (val_time_count, N) -> map to val_samples whose targets range val_time_start..val_time_end-1
    # Build mapping: val_samples' targets indexes are sample_target_timeidx[val_samples] which should be contiguous range equal to val_time_start..val_time_end-1
    target_times = sample_target_timeidx[val_samples]
    # index in sar_preds corresponds to i = target_times - train_time_end (assuming contiguous)
    idxs = (target_times - train_time_end).astype(int)
    sar_for_samples = sar_preds[idxs, :]
    oof_preds['sarimax'][val_samples,:] = sar_for_samples
    print("SARIMAX oof filled")

    # ------- (E) Prophet per-node forecast (if available) -------
    if HAS_PROPHET:
        prop_preds = prophet_forecast_for_fold(piv_density_orig, period_vec, train_time_end, val_time_end)
        idxs = (target_times - train_time_end).astype(int)
        p_for_samples = prop_preds[idxs, :]
        oof_preds['prophet'][val_samples,:] = p_for_samples
        print("Prophet oof filled")
    else:
        print("Prophet not available; skipping.")


=== FOLD 1/3 | train samples 239 val samples 365 ===
train_time_end idx: 273 val_time range: 274 639
LSTM oof filled
STGCN oof filled
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.061775 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1729
[LightGBM] [Info] Number of data points in the train set: 47561, number of used features: 7
[LightGBM] [Info] Start training from score 145.775401
LGBM oof filled
XGB oof filled
SARIMAX oof filled


00:34:57 - cmdstanpy - INFO - Chain [1] start processing
00:34:58 - cmdstanpy - INFO - Chain [1] done processing
00:35:00 - cmdstanpy - INFO - Chain [1] start processing
00:35:00 - cmdstanpy - INFO - Chain [1] done processing
00:35:01 - cmdstanpy - INFO - Chain [1] start processing
00:35:02 - cmdstanpy - INFO - Chain [1] done processing
00:35:04 - cmdstanpy - INFO - Chain [1] start processing
00:35:05 - cmdstanpy - INFO - Chain [1] done processing
00:35:06 - cmdstanpy - INFO - Chain [1] start processing
00:35:06 - cmdstanpy - INFO - Chain [1] done processing
00:35:07 - cmdstanpy - INFO - Chain [1] start processing
00:35:07 - cmdstanpy - INFO - Chain [1] done processing
00:35:08 - cmdstanpy - INFO - Chain [1] start processing
00:35:09 - cmdstanpy - INFO - Chain [1] done processing
00:35:10 - cmdstanpy - INFO - Chain [1] start processing
00:35:10 - cmdstanpy - INFO - Chain [1] done processing
00:35:11 - cmdstanpy - INFO - Chain [1] start processing
00:35:11 - cmdstanpy - INFO - Chain [1]

Prophet oof filled

=== FOLD 2/3 | train samples 604 val samples 365 ===
train_time_end idx: 638 val_time range: 639 1004
LSTM oof filled
STGCN oof filled
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.008448 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1729
[LightGBM] [Info] Number of data points in the train set: 120196, number of used features: 7
[LightGBM] [Info] Start training from score 147.139580
LGBM oof filled
XGB oof filled
SARIMAX oof filled


00:47:47 - cmdstanpy - INFO - Chain [1] start processing
00:47:47 - cmdstanpy - INFO - Chain [1] done processing
00:47:48 - cmdstanpy - INFO - Chain [1] start processing
00:47:48 - cmdstanpy - INFO - Chain [1] done processing
00:47:49 - cmdstanpy - INFO - Chain [1] start processing
00:47:49 - cmdstanpy - INFO - Chain [1] done processing
00:47:49 - cmdstanpy - INFO - Chain [1] start processing
00:47:50 - cmdstanpy - INFO - Chain [1] done processing
00:47:50 - cmdstanpy - INFO - Chain [1] start processing
00:47:50 - cmdstanpy - INFO - Chain [1] done processing
00:47:51 - cmdstanpy - INFO - Chain [1] start processing
00:47:51 - cmdstanpy - INFO - Chain [1] done processing
00:47:52 - cmdstanpy - INFO - Chain [1] start processing
00:47:52 - cmdstanpy - INFO - Chain [1] done processing
00:47:52 - cmdstanpy - INFO - Chain [1] start processing
00:47:53 - cmdstanpy - INFO - Chain [1] done processing
00:47:53 - cmdstanpy - INFO - Chain [1] start processing
00:47:54 - cmdstanpy - INFO - Chain [1]

Prophet oof filled

=== FOLD 3/3 | train samples 969 val samples 353 ===
train_time_end idx: 1003 val_time range: 1004 1357
LSTM oof filled
STGCN oof filled
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.006295 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1729
[LightGBM] [Info] Number of data points in the train set: 192831, number of used features: 7
[LightGBM] [Info] Start training from score 141.957315
LGBM oof filled
XGB oof filled


01:00:44 - cmdstanpy - INFO - Chain [1] start processing


SARIMAX oof filled


01:00:44 - cmdstanpy - INFO - Chain [1] done processing
01:00:45 - cmdstanpy - INFO - Chain [1] start processing
01:00:45 - cmdstanpy - INFO - Chain [1] done processing
01:00:45 - cmdstanpy - INFO - Chain [1] start processing
01:00:45 - cmdstanpy - INFO - Chain [1] done processing
01:00:45 - cmdstanpy - INFO - Chain [1] start processing
01:00:45 - cmdstanpy - INFO - Chain [1] done processing
01:00:46 - cmdstanpy - INFO - Chain [1] start processing
01:00:46 - cmdstanpy - INFO - Chain [1] done processing
01:00:46 - cmdstanpy - INFO - Chain [1] start processing
01:00:46 - cmdstanpy - INFO - Chain [1] done processing
01:00:46 - cmdstanpy - INFO - Chain [1] start processing
01:00:47 - cmdstanpy - INFO - Chain [1] done processing
01:00:47 - cmdstanpy - INFO - Chain [1] start processing
01:00:47 - cmdstanpy - INFO - Chain [1] done processing
01:00:47 - cmdstanpy - INFO - Chain [1] start processing
01:00:47 - cmdstanpy - INFO - Chain [1] done processing
01:00:47 - cmdstanpy - INFO - Chain [1] 

Prophet oof filled


In [11]:
# ------------------ 8) Build meta training matrix from OOF predictions ------------------
# Meta uses first-horizon OOF predictions from each base model as features, flattened per sample-node row
oof_list = ['lstm','stgcn','lgbm','xgb','sarimax','prophet']
oof_arrays = []
for k in oof_list:
    arr = oof_preds[k]  # (S,N)
    oof_arrays.append(arr.reshape(-1))  # flatten sample-node

meta_X_full = np.stack(oof_arrays, axis=1)   # (S*N, n_models)
y_full = Y_orig[:,0,:].reshape(-1)           # target flattened (original units)

# mask rows where any NaN in meta features
valid_mask = ~np.isnan(meta_X_full).any(axis=1)
print("Meta rows total:", len(y_full), "valid rows:", valid_mask.sum())

X_meta = meta_X_full[valid_mask]
y_meta = y_full[valid_mask]

# Train meta learner (Ridge regression)
meta_model = Ridge(alpha=1.0)
meta_model.fit(X_meta, y_meta)
print("Meta trained. Coefs:", meta_model.coef_)

# Evaluate meta on OOF (using valid_mask)
oof_meta_preds = np.full_like(y_full, np.nan)
oof_meta_preds[valid_mask] = meta_model.predict(X_meta)
mae_meta = mean_absolute_error(y_full[valid_mask], oof_meta_preds[valid_mask])
rmse_meta = np.sqrt(mean_squared_error(y_full[valid_mask], oof_meta_preds[valid_mask]))
r2_meta = r2_score(y_full[valid_mask], oof_meta_preds[valid_mask])
print("\n=== OOF Meta Metrics ===")
print(f"MAE: {mae_meta:.4f} | RMSE: {rmse_meta:.4f} | R2: {r2_meta:.4f}")

# Save models / artifacts
joblib.dump({'meta': meta_model}, os.path.join(OUT_DIR, "meta_model.joblib"))
print("Artifacts saved to", OUT_DIR)

Meta rows total: 263078 valid rows: 215517
Meta trained. Coefs: [ 0.3482563   0.06001833  0.6098419  -0.00569794 -0.23524441  0.18495705]

=== OOF Meta Metrics ===
MAE: 31.8776 | RMSE: 75.7841 | R2: 0.9123
Artifacts saved to hybrid_oof_outputs
